In [1]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

Cloning into 'temporalgnn-nids'...
remote: Enumerating objects: 736, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (285/285), done.
remote: Total 736 (delta 156), reused 226 (delta 71), pack-reused 276 (from 1)
Receiving objects: 100% (736/736), 4.43 MiB | 15.02 MiB/s, done.
Resolving deltas: 100% (230/230), done.
Mounted at /content/drive


In [2]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.0 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import copy
import random
import json
import time

import torch
import torch.nn as nn

from torch_geometric.loader import DataLoader


In [4]:
from utils.datasets      import NF_IDS_Dataset
from utils.models        import StaticGNN_Identity_Entropy, ST_GNN_Identity_Entropy
from utils.metrics       import calculate_metrics_gnn
from utils.training      import train_epoch, evaluate, find_optimal_threshold, set_seed, run_multiple_seeds
from utils.experiment    import ExperimentManager, EarlyStopping, NumpyEncoder
from utils.visualization import save_plots


# Auxiliary

In [5]:
ROOT_PATH = "./dataset_processed_entropy"

In [6]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [7]:
ROOT_DIR = "./results_earlystopping_entropy"


LOGS_DIR = os.path.join(ROOT_DIR, "logs")
PLOTS_DIR = os.path.join(ROOT_DIR, "plots")
MODELS_DIR = os.path.join(ROOT_DIR, "saved_models")

# Main

## Seeds

### StaticGNN_Identity

In [8]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 7 dst_port + 5 protocol + 20 numeric
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [9]:
model_config = {
    "model_name": None,
    "type": "SaticGNN_Identity_Entropy",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [10]:
EXPERIMENT_NAME="StaticGNN_BiasOn_robust_Identity_entropy"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [12]:
run_multiple_seeds(
    model_class=StaticGNN_Identity_Entropy,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)

 Starting Multi-Seed Run: StaticGNN_BiasOn_robust_Identity_entropy
   Seeds: [42, 123, 777, 2024, 99]
------------------------------------------------------------

Running seed 42 | run_id=StaticGNN_BiasOn_robust_Identity_entropy_seed42_20260417_105722

StaticGNN_BiasOn_robust_Identity_entropy_seed42
   Ep 1 | Loss: 0.2042 | Val Loss: 0.2471 | Val AUC-PR: 0.0621 (*)
   Ep 2 | Loss: 0.1849 | Val Loss: 0.2285 | Val AUC-PR: 0.1305 (*)
   Ep 10 | Loss: 0.1728 | Val Loss: 0.2064 | Val AUC-PR: 0.1341 (*)
   Ep 13 | Loss: 0.1712 | Val Loss: 0.2027 | Val AUC-PR: 0.1396 (*)
   Ep 14 | Loss: 0.1631 | Val Loss: 0.2035 | Val AUC-PR: 0.1420 (*)
   Ep 15 | Loss: 0.1635 | Val Loss: 0.1955 | Val AUC-PR: 0.2032 (*)
   Ep 17 | Loss: 0.1584 | Val Loss: 0.1963 | Val AUC-PR: 0.2297 (*)
   Ep 18 | Loss: 0.1569 | Val Loss: 0.1951 | Val AUC-PR: 0.2373 (*)
   Ep 19 | Loss: 0.1571 | Val Loss: 0.1943 | Val AUC-PR: 0.2916 (*)
   Ep 20 | Loss: 0.1557 | Val Loss: 0.1900 | Val AUC-PR: 0.2696 
   Ep 23 | Loss: 0.1518

### ST_GNN_Identity

In [ ]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 7 dst_port + 5 protocol + 20 numeric
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [ ]:
model_config = {
    "model_name": None,
    "type": "ST_GNN_Identity_Entropy",
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

In [ ]:
EXPERIMENT_NAME="ST_GNN_BiasOn_robust_Identity_clone_entropy"

exp_manager = ExperimentManager(log_file=os.path.join(LOGS_DIR, EXPERIMENT_NAME, f"run_metrics_{EXPERIMENT_NAME}.csv"), model_dir=os.path.join(MODELS_DIR, EXPERIMENT_NAME))

In [ ]:
run_multiple_seeds(
    model_class=ST_GNN_Identity_Entropy,
    model_config=model_config,
    train_loader=train_loader,
    val_loader=val_loader,
    manager=exp_manager,
    seeds=[42, 123, 777, 2024, 99],
    epochs=EPOCHS,
    device=DEVICE,
    experiment_name=EXPERIMENT_NAME,
    json_dir=LOGS_DIR,
    plots_dir=PLOTS_DIR
)